# serve

> Put a rishi model behind an OpenAI-compatible endpoint, so agent harnesses that expect a hosted LLM - `buzz-agent`, and anything else that speaks Chat Completions - can drive a model running on your own machine.

In [ ]:
#| default_exp serve

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import json, os, time, uuid, threading, argparse
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from fastcore.all import store_attr, L, ifnone
from litert_lm import Message, Contents, ToolCall, Backend
from litert_lm._messages import Text, ToolResponse, normalize_message
from rishi.core import Chat, resp_text, thought, gemma4_e2b
from rishi.mcp import SchemaTool

## Why an HTTP endpoint

`Chat` is the right interface when *you* drive the model. An agent harness drives it the other way round: it owns the loop, the tools, and the history, and it wants an LLM it can POST to. `buzz-agent` is one of those - it takes `OPENAI_COMPAT_BASE_URL` and speaks Chat Completions to whatever is listening.

So the whole integration is a translation layer. Two things make it small: buzz sends tool definitions in exactly the shape litert wants, and litert can hand tool calls back instead of running them (`automatic_tool_calling=False`), which is what a harness that owns its own tools needs.

## Requests in

One wrinkle: the harness is stateless and resends the whole conversation every call, while a litert `Conversation` is stateful. Each request therefore builds a fresh conversation whose preface is the history, and sends the final message to generate against.

In [ ]:
#| export
def _txt(c):
    "Text of an OpenAI `content` field, which is either a string or a list of content parts."
    if c is None: return ''
    if isinstance(c, str): return c
    if isinstance(c, list): return ''.join(p.get('text', '') for p in c
                                           if isinstance(p, dict) and p.get('type') in ('text', 'input_text'))
    return str(c)

def _args(a):
    "OpenAI tool-call arguments, which arrive as a JSON *string*."
    if isinstance(a, dict): return a
    try: return json.loads(a or '{}')
    except json.JSONDecodeError: return {}

In [ ]:
#| export
def oai_msgs(msgs):
    "OpenAI chat messages -> litert `Message`s, keeping each tool result paired with its call."
    out, names = [], {}
    for m in msgs or []:
        role, txt = m.get('role', 'user'), _txt(m.get('content'))
        if role == 'tool':
            name = names.get(m.get('tool_call_id')) or m.get('name') or 'tool'
            out.append(Message.tool(Contents([ToolResponse(name, txt)])))
        elif role == 'assistant':
            tcs = m.get('tool_calls') or []
            for tc in tcs: names[tc.get('id')] = tc.get('function', {}).get('name', '')
            out.append(Message.model(Contents([Text(txt)]) if txt else None,
                tool_calls=[ToolCall(tc.get('function', {}).get('name', ''),
                                     _args(tc.get('function', {}).get('arguments'))) for tc in tcs]))
        elif role == 'system': out.append(Message.system(txt))
        else: out.append(Message.user(txt))
    return out

In [ ]:
ms = oai_msgs([{'role':'system','content':'be brief'},
               {'role':'user','content':[{'type':'text','text':'list files'}]},
               {'role':'assistant','content':'sure','tool_calls':[
                   {'id':'call_1','type':'function','function':{'name':'shell','arguments':'{"command":"ls"}'}}]},
               {'role':'tool','tool_call_id':'call_1','content':'app.py'}])
test_eq([m.to_json()['role'] for m in ms], ['system','user','model','tool'])
test_eq(ms[2].to_json()['tool_calls'][0]['function'], {'name':'shell','arguments':{'command':'ls'}})
test_eq(ms[3].to_json()['content'][0], {'type':'tool_response','name':'shell','response':'app.py'})

The tool result carries only a `tool_call_id`, so the name is recovered from the assistant message that requested it - which is why `oai_msgs` walks the history in order rather than mapping each message independently.

In [ ]:
#| export
def oai_tools(tools):
    "OpenAI tool definitions -> litert tools. The harness runs them, so these have no implementation."
    return L(tools or []).map(lambda t: SchemaTool((f := t.get('function', t)).get('name', ''),
                                                   f.get('description', ''), f.get('parameters')))

In [ ]:
ts = oai_tools([{'type':'function','function':{'name':'shell','description':'Run it.',
                 'parameters':{'type':'object','properties':{'command':{'type':'string'}}}}}])
test_eq(ts[0].get_tool_description()['function']['name'], 'shell')
test_fail(lambda: ts[0].execute({}), contains='no implementation')

## Responses out

`buzz-agent` reads `choices[0].message`, expects tool-call arguments as a JSON *string*, and looks for reasoning on `reasoning_content` - so the model's thinking channel has somewhere to go.

In [ ]:
#| export
def _mkid(): return 'call_' + uuid.uuid4().hex[:24]

def mk_tool_calls(resp, mkid=_mkid):
    "litert tool calls -> OpenAI `tool_calls`, whose arguments are JSON-encoded strings."
    return [{'id': mkid(), 'type': 'function',
             'function': {'name': (f := tc.get('function', {})).get('name', ''),
                          'arguments': json.dumps(f.get('arguments', {}))}}
            for tc in (resp.get('tool_calls') or [])]

def mk_completion(resp, model='rishi', usage=None, mkid=_mkid):
    "A litert response -> an OpenAI chat-completion object."
    tcs, txt, th = mk_tool_calls(resp, mkid), resp_text(resp), thought(resp)
    msg = {'role': 'assistant', 'content': txt or None}
    if tcs: msg['tool_calls'] = tcs
    if th: msg['reasoning_content'] = th
    fin = 'tool_calls' if tcs else ('length' if resp.get('truncated') else 'stop')
    return {'id': 'chatcmpl-' + uuid.uuid4().hex[:24], 'object': 'chat.completion',
            'created': int(time.time()), 'model': model,
            'choices': [{'index': 0, 'message': msg, 'finish_reason': fin}],
            'usage': ifnone(usage, {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0})}

In [ ]:
r = mk_completion({'content':[{'type':'text','text':'hi'}]}, model='gemma')
test_eq(r['choices'][0]['finish_reason'], 'stop')
test_eq(r['choices'][0]['message'], {'role':'assistant','content':'hi'})

r = mk_completion({'content':[], 'tool_calls':[{'function':{'name':'shell','arguments':{'command':'ls'}}}]},
                  mkid=lambda: 'call_x')
test_eq(r['choices'][0]['finish_reason'], 'tool_calls')
test_eq(r['choices'][0]['message']['tool_calls'],
        [{'id':'call_x','type':'function','function':{'name':'shell','arguments':'{"command": "ls"}'}}])
test_eq(r['choices'][0]['message']['content'], None)

r = mk_completion({'content':[{'type':'text','text':'a'}], 'channels':{'thought':'hmm'}, 'truncated':True})
test_eq(r['choices'][0]['finish_reason'], 'length')
test_eq(r['choices'][0]['message']['reasoning_content'], 'hmm')

## The completer

`Completer` is the whole translation, independent of HTTP: a request dict in, a completion dict out. One engine serves every request, serialised by a lock, because a litert engine is a single model in memory rather than a pool.

In [ ]:
#| export
class Completer:
    "Answers OpenAI chat-completion requests from one litert engine."
    def __init__(self,
        engine,                    # a litert `Engine` (or anything with its `create_conversation`/`tokenize`)
        model:str='rishi',         # name echoed back in responses
        think:bool=False,          # enable the model's thinking channel
        filter_think:bool=True,    # keep thinking out of the KV cache
        max_output_tokens:int=None,# cap when the request does not set one
        sampler_config=None        # litert `SamplerConfig`
    ):
        store_attr(); self.lock = threading.Lock()

    def conv_kw(self, req):
        "Conversation kwargs for one request."
        kw = dict(tools=list(oai_tools(req.get('tools'))) or None, automatic_tool_calling=False,
                  filter_channel_content_from_kv_cache=self.filter_think, sampler_config=self.sampler_config)
        if self.think: kw['extra_context'] = {'enable_thinking': True}
        return kw

    def __call__(self, req):
        "Run one chat-completion request."
        msgs = oai_msgs(req.get('messages'))
        if not msgs: raise ValueError('request has no messages')
        *head, last = msgs
        mx = req.get('max_completion_tokens') or req.get('max_tokens') or self.max_output_tokens
        with self.lock:
            with self.engine.create_conversation(
                    messages=[normalize_message(m) for m in head] or None, **self.conv_kw(req)) as conv:
                resp = conv.send_message(last, max_output_tokens=mx)
                out = len(self.engine.tokenize(resp_text(resp)))
                total = conv.token_count
        usage = {'prompt_tokens': max(total - out, 0), 'completion_tokens': out, 'total_tokens': total}
        return mk_completion(resp, ifnone(req.get('model'), self.model), usage)

### Testing the wiring without a model

Loading gemma costs gigabytes and seconds, which is a poor fit for checking that a harness is plumbed in correctly. `ScriptedEngine` stands in for a litert engine and replays canned replies, so the wire path can be exercised on its own - the same trick `buzz-agent` uses with its fake LLM. It is also the honest way to test a *translation* layer: what matters is the JSON, not the sampling.

In [ ]:
#| export
def mk_reply(text='', tool_calls=None, thinking=''):
    "A litert-shaped response dict, for scripting a `ScriptedEngine`."
    r = {'role': 'assistant', 'content': [{'type': 'text', 'text': text}] if text else []}
    if tool_calls: r['tool_calls'] = [{'type': 'function', 'function': {'name': n, 'arguments': a}}
                                      for n, a in tool_calls]
    if thinking: r['channels'] = {'thought': thinking}
    return r

class _ScriptedConv:
    def __init__(self, eng, kw): self.eng, self.kw, self.token_count = eng, kw, 0
    def __enter__(self): return self
    def __exit__(self, *a): return False
    def send_message(self, msg, max_output_tokens=None):
        self.eng.sent.append((normalize_message(msg), max_output_tokens))
        self.token_count = 10 * len(self.eng.sent) + len(self.kw.get('messages') or [])
        rs = self.eng.replies
        r = rs.pop(0) if len(rs) > 1 else (rs[0] if rs else mk_reply('ok'))
        return r(msg) if callable(r) else r

class ScriptedEngine:
    "A stand-in for a litert `Engine` that replays scripted replies, for testing wiring without a model."
    def __init__(self, replies=None):
        self.replies = list(replies or [mk_reply('ok')])
        self.convs, self.sent = [], []
    def create_conversation(self, **kw):
        self.convs.append(kw); return _ScriptedConv(self, kw)
    def tokenize(self, text): return (text or '').split()

In [ ]:
eng = ScriptedEngine([mk_reply('hello there')])
c = Completer(eng, model='gemma4-e2b')
out = c({'messages': [{'role':'system','content':'be brief'}, {'role':'user','content':'hi'}],
         'max_completion_tokens': 64})
test_eq(out['choices'][0]['message']['content'], 'hello there')
test_eq(out['model'], 'gemma4-e2b')
test_eq(out['usage']['completion_tokens'], 2)
test_eq(eng.sent[0][1], 64)                                   # max tokens forwarded
test_eq(eng.sent[0][0]['role'], 'user')                       # last message is the one sent
test_eq([m['role'] for m in eng.convs[0]['messages']], ['system'])  # the rest is the preface
test_eq(eng.convs[0]['automatic_tool_calling'], False)        # the harness runs the tools, not litert

# a request carrying tools comes back as a tool call the harness can execute
eng = ScriptedEngine([mk_reply(tool_calls=[('shell', {'command': 'ls'})])])
out = Completer(eng)({'messages': [{'role':'user','content':'list files'}],
                      'tools': [{'type':'function','function':{'name':'shell','parameters':{}}}]})
test_eq(out['choices'][0]['finish_reason'], 'tool_calls')
test_eq(json.loads(out['choices'][0]['message']['tool_calls'][0]['function']['arguments']), {'command':'ls'})
test_eq([t.name for t in eng.convs[0]['tools']], ['shell'])

## The server

A small stdlib HTTP server: `POST /v1/chat/completions`, plus `/v1/models` and `/health` so harnesses and humans can check it is alive. Paths match on suffix, because base URLs are written both with and without the `/v1`.

In [ ]:
#| export
MAX_BODY = 32 * 1024 * 1024

class RishiHandler(BaseHTTPRequestHandler):
    "OpenAI-compatible HTTP surface over a `Completer`."
    protocol_version = 'HTTP/1.1'
    def _reply(self, code, o):
        body = json.dumps(o).encode()
        self.send_response(code)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _err(self, code, msg, typ='invalid_request_error'):
        self._reply(code, {'error': {'message': msg, 'type': typ}})

    def do_GET(self):
        p = self.path.rstrip('/')
        if p.endswith('/models'):
            m = self.server.completer.model
            return self._reply(200, {'object': 'list', 'data': [{'id': m, 'object': 'model', 'owned_by': 'rishi'}]})
        if p.endswith('/health'): return self._reply(200, {'status': 'ok'})
        self._err(404, f'no route for {self.path}')

    def do_POST(self):
        if not self.path.rstrip('/').endswith('/chat/completions'):
            return self._err(404, f'no route for {self.path}')
        n = int(self.headers.get('Content-Length') or 0)
        if n > MAX_BODY: return self._err(413, f'body over {MAX_BODY} bytes')
        try: req = json.loads(self.rfile.read(n) or b'{}')
        except json.JSONDecodeError as e: return self._err(400, f'invalid JSON: {e}')
        try: out = self.server.completer(req)
        except Exception as e:
            self.log_error('completion failed: %s: %s', type(e).__name__, e)
            return self._err(500, f'{type(e).__name__}: {e}', 'server_error')
        self._reply(200, out)

    def log_message(self, fmt, *a):
        if getattr(self.server, 'verbose', False): super().log_message(fmt, *a)

In [ ]:
#| export
def mk_server(completer, host='127.0.0.1', port=8017, verbose=False):
    "An HTTP server exposing `completer`; call `serve_forever()` on it, or use `serve`."
    httpd = ThreadingHTTPServer((host, port), RishiHandler)
    httpd.completer, httpd.verbose, httpd.daemon_threads = completer, verbose, True
    return httpd

def serve(engine=None, model_id=gemma4_e2b, host='127.0.0.1', port=8017, background=False,
          verbose=False, model=None, think=False, max_output_tokens=None, eng_kw=None, **kw):
    "Serve a rishi model at `http://host:port/v1`. Returns the server (already running if `background`)."
    if engine is None: engine = Chat.create_engine(model_id, **(eng_kw or {}))
    c = Completer(engine, model=ifnone(model, model_id), think=think, max_output_tokens=max_output_tokens, **kw)
    httpd = mk_server(c, host, port, verbose)
    if background: threading.Thread(target=httpd.serve_forever, daemon=True).start()
    else:
        print(f'rishi serving {c.model} on http://{host}:{port}/v1 - ^C to stop')
        try: httpd.serve_forever()
        except KeyboardInterrupt: httpd.shutdown(); httpd.server_close()
    return httpd

End to end over real HTTP, still with no model loaded:

In [ ]:
#| hide
import urllib.request
eng = ScriptedEngine([mk_reply('pong', thinking='thinking about it')])
srv = serve(engine=eng, model='gemma4-e2b', port=8123, background=True)

def _post(path, o):
    r = urllib.request.urlopen(urllib.request.Request(f'http://127.0.0.1:8123{path}',
        data=json.dumps(o).encode(), headers={'Content-Type':'application/json'}))
    return json.loads(r.read())

out = _post('/v1/chat/completions', {'messages':[{'role':'user','content':'ping'}]})
test_eq(out['choices'][0]['message']['content'], 'pong')
test_eq(out['choices'][0]['message']['reasoning_content'], 'thinking about it')
test_eq(json.loads(urllib.request.urlopen('http://127.0.0.1:8123/v1/models').read())['data'][0]['id'], 'gemma4-e2b')
test_fail(lambda: _post('/v1/nope', {}), contains='404')
srv.shutdown(); srv.server_close()

## From the command line

`rishi-serve` starts the endpoint without writing any Python, which is what you want when the thing driving the model is a separate process.

In [ ]:
#| export
def main():
    "Console entry point: `rishi-serve --port 8017`."
    p = argparse.ArgumentParser(prog='rishi-serve', description='Serve a rishi model over an OpenAI-compatible API.')
    p.add_argument('--model-id', default=gemma4_e2b, help='huggingface model id')
    p.add_argument('--model-path', default=None, help='local .litertlm path')
    p.add_argument('--host', default='127.0.0.1'); p.add_argument('--port', type=int, default=8017)
    p.add_argument('--cache-dir', default=None, help='engine cache dir (required for the GPU backend)')
    p.add_argument('--gpu', action='store_true', help='use the GPU backend')
    p.add_argument('--think', action='store_true', help='enable the thinking channel')
    p.add_argument('--max-output-tokens', type=int, default=None)
    p.add_argument('--verbose', action='store_true')
    a = p.parse_args()
    eng = Chat.create_engine(a.model_id, a.model_path, Backend.GPU() if a.gpu else Backend.CPU(),
                             multimodal=False, cache_dir=a.cache_dir)
    print(f"point a harness at it with:\n"
          f"  BUZZ_AGENT_PROVIDER=openai OPENAI_COMPAT_API=chat OPENAI_COMPAT_API_KEY=local \\\n"
          f"  OPENAI_COMPAT_MODEL={a.model_id} OPENAI_COMPAT_BASE_URL=http://{a.host}:{a.port}/v1")
    serve(engine=eng, model=a.model_id, host=a.host, port=a.port, think=a.think,
          max_output_tokens=a.max_output_tokens, verbose=a.verbose)

## Driving it from buzz-agent

With the endpoint up, `buzz-agent` needs four environment variables and an MCP server to act through. `OPENAI_COMPAT_API=chat` matters: left on `auto` it picks the Responses dialect for `*.openai.com` hosts and Chat Completions everywhere else, and being explicit costs nothing.

```sh
rishi-serve --port 8017 --cache-dir .cache/litertlm &

BUZZ_AGENT_PROVIDER=openai \
OPENAI_COMPAT_API=chat \
OPENAI_COMPAT_API_KEY=local \
OPENAI_COMPAT_MODEL=gemma4-e2b \
OPENAI_COMPAT_BASE_URL=http://127.0.0.1:8017/v1 \
BUZZ_AGENT_MAX_ROUNDS=8 \
  buzz-agent
```

`buzz-agent` speaks ACP on stdin, so a client (Zed, `buzz-acp`, or a handful of JSON-RPC lines) opens a session, names the MCP servers to spawn, and prompts. The model decides; buzz runs the tools and keeps the loop bounded.

A local model this small will not hold up as a general coding agent - it has a short context and it drops tool-calling syntax under pressure. What it is good for is the narrow, repetitive work you would rather not send anywhere: renaming things across a repo, summarising a diff, triaging a log. Give it `BUZZ_AGENT_MAX_ROUNDS` and a small tool set and it stays useful.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()